# Quantum Optimization Experiment Analysis
## Comprehensive Analysis of Energy Errors, Circuit Metrics, and Mitigation Performance

This notebook analyzes the quantum optimization experiment results from `exp_20260522_115806/merged_results.csv`, focusing on:
- Energy error differences (noisy vs ideal, mitigated vs ideal)
- Mitigation effectiveness across different factories
- Circuit complexity metrics (depth, active qubits)
- Computational cost analysis
- Problem parameter sensitivity

## 1. Load and Explore the Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
csv_path = '/home/gibbo/Documents/PhD/Quantum/spindox/easy-q/quantum_optimization/experiments_automation/exp_20260522_115806/merged_results.csv'
df = pd.read_csv(csv_path)

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nFirst few rows:")
df.head()

Dataset Shape: (155, 57)

Column Names:
['label', 'seed', 'problem_class', 'circuit_class', 'layers', 'backend', 'logic_qubits', 'backend_qubits', 'mitigation_technique', 'num_estimator_shots', 'ideal_energy', 'noisy_energy', 'clifford_ideal_energy', 'clifford_noisy_energy', 'clifford_mitigated_energy', 'error_pct_noisy', 'error_pct_mitigated', 'circuit_creation_time', 'transpilation_time', 'transpilation_clifford_time', 'ideal_estimation_time', 'noisy_estimation_time', 'clifford_ideal_estimation_time', 'clifford_noisy_estimation_time', 'clifford_mitigated_estimation_time', 'zne_factory_plot', 'problem_params.n_nodes', 'problem_params.density', 'problem_params.num_groups', 'problem_params.num_instruments_per_group', 'problem_params.num_machines', 'problem_params.num_products', 'virtual_qc_metrics.depth', 'virtual_qc_metrics.2q_gates', 'virtual_qc_metrics.clifford_gates', 'virtual_qc_metrics.non_clifford_gates', 'virtual_qc_metrics.t_gates', 'virtual_qc_metrics.total_gates', 'virtual_qc

,label,seed,problem_class,circuit_class,layers,backend,logic_qubits,backend_qubits,mitigation_technique,num_estimator_shots,...,clifford_qc_metrics.2q_gates,clifford_qc_metrics.clifford_gates,clifford_qc_metrics.non_clifford_gates,clifford_qc_metrics.t_gates,clifford_qc_metrics.total_gates,clifford_qc_metrics.num_active_qubits,performance.only_clifford,performance.pruned_noise,mitigation_params.factory,mitigation_params.noise_values
0,M2_P10_S111_Flinear_out,111,SimpleProductionPlanningProblem,AncillaQAOACircuit,2,ibm_marrakesh,26,156,zne,1024,...,2356,8938,0,0,8938,41,True,False,linear,"[1, 2, 3]"
1,M2_P10_S111_Frichardson_out,111,SimpleProductionPlanningProblem,AncillaQAOACircuit,2,ibm_marrakesh,26,156,zne,1024,...,2356,8915,0,0,8915,41,True,False,richardson,"[1, 2, 3]"
2,M2_P10_S222_Fadaexp_out,222,SimpleProductionPlanningProblem,AncillaQAOACircuit,2,ibm_marrakesh,26,156,zne,1024,...,2455,9120,0,0,9120,37,True,False,adaexp,"[1, 2, 3]"
3,M2_P10_S222_Flinear_out,222,SimpleProductionPlanningProblem,AncillaQAOACircuit,2,ibm_marrakesh,26,156,zne,1024,...,2455,9120,0,0,9120,37,True,False,linear,"[1, 2, 3]"
4,M2_P10_S222_Frichardson_out,222,SimpleProductionPlanningProblem,AncillaQAOACircuit,2,ibm_marrakesh,26,156,zne,1024,...,2455,9129,0,0,9129,37,True,False,richardson,"[1, 2, 3]"


In [2]:
# Check for missing values and basic statistics
print("Missing values per column:")
print(df.isnull().sum())
print("\n" + "="*80 + "\n")

# Display key columns
key_columns = ['seed', 'problem_params.num_machines', 'problem_params.num_products',
               'clifford_ideal_energy', 'clifford_noisy_energy', 'clifford_mitigated_energy',
               'error_pct_noisy', 'error_pct_mitigated', 'mitigation_params.factory',
               'clifford_qc_metrics.depth', 'clifford_qc_metrics.num_active_qubits']

print("Basic statistics for key columns:")
print(df[key_columns].describe())
print("\n" + "="*80 + "\n")
print("Unique values:")
print(f"Num Machines: {sorted(df['problem_params.num_machines'].unique())}")
print(f"Num Products: {sorted(df['problem_params.num_products'].unique())}")
print(f"Seeds: {sorted(df['seed'].unique())}")
print(f"Factories: {df['mitigation_params.factory'].unique()}")

Missing values per column:
label                                         0
seed                                          0
problem_class                                 0
circuit_class                                 0
layers                                        0
backend                                       0
logic_qubits                                  0
backend_qubits                                0
mitigation_technique                          0
num_estimator_shots                           0
ideal_energy                                155
noisy_energy                                155
clifford_ideal_energy                         0
clifford_noisy_energy                         0
clifford_mitigated_energy                     0
error_pct_noisy                               0
error_pct_mitigated                           0
circuit_creation_time                         0
transpilation_time                            0
transpilation_clifford_time                   0
ideal_estimat

## 2. Data Preprocessing and Aggregation

In [2]:
# Rename columns for easier access
df_clean = df.copy()
df_clean['num_machines'] = df_clean['problem_params.num_machines']
df_clean['num_products'] = df_clean['problem_params.num_products']
df_clean['ideal_energy'] = df_clean['clifford_ideal_energy']
df_clean['noisy_energy'] = df_clean['clifford_noisy_energy']
df_clean['mitigated_energy'] = df_clean['clifford_mitigated_energy']
df_clean['error_noisy'] = df_clean['error_pct_noisy']
df_clean['error_mitigated'] = df_clean['error_pct_mitigated']
df_clean['depth_clifford'] = df_clean['clifford_qc_metrics.depth']
df_clean['active_qubits_clifford'] = df_clean['clifford_qc_metrics.num_active_qubits']
df_clean['depth_transpiled'] = df_clean['transpiled_qc_metrics.depth']
df_clean['active_qubits_transpiled'] = df_clean['transpiled_qc_metrics.num_active_qubits']
df_clean['factory'] = df_clean['mitigation_params.factory']

# Calculate energy differences from ideal
df_clean['energy_error_noisy'] = (df_clean['noisy_energy'] - df_clean['ideal_energy']).abs()
df_clean['energy_error_mitigated'] = (df_clean['mitigated_energy'] - df_clean['ideal_energy']).abs()
df_clean['energy_improvement'] = df_clean['energy_error_noisy'] - df_clean['energy_error_mitigated']

# Create parameter combination column
df_clean['param_combo'] = df_clean.apply(lambda x: f"M{int(x['num_machines'])}_P{int(x['num_products'])}", axis=1)

print("Data preprocessing complete!")
print(f"Original columns: {len(df.columns)}")
print(f"Useful columns for analysis: {len([c for c in df_clean.columns if not c.startswith('problem_params.') and not c.startswith('virtual_qc') and not c.startswith('transpiled_qc') and not c.startswith('clifford_qc') and not c.startswith('mitigation_params.')])}")

# Group by parameter combinations and factory
print("\nGrouping data by (num_machines, num_products) combinations:")
grouped = df_clean.groupby(['num_machines', 'num_products', 'factory']).size().reset_index(name='count')
print(grouped)

Data preprocessing complete!
Original columns: 57
Useful columns for analysis: 42

Grouping data by (num_machines, num_products) combinations:
    num_machines  num_products     factory  count
0              2             6      adaexp      3
1              2             6      linear      5
2              2             6  richardson      5
3              2             8      adaexp      3
4              2             8      linear      5
5              2             8  richardson      5
6              2            10      adaexp      2
7              2            10      linear      5
8              2            10  richardson      5
9              2            12      adaexp      2
10             2            12      linear      5
11             2            12  richardson      5
12             3             6      adaexp      2
13             3             6      linear      5
14             3             6  richardson      5
15             3             8      adaexp      2
16     

In [3]:
# Aggregate data by parameter combinations (grouping by num_machines and num_products)
agg_data = df_clean.groupby(['num_machines', 'num_products']).agg({
    'ideal_energy': 'mean',
    'noisy_energy': 'mean',
    'mitigated_energy': 'mean',
    'error_noisy': 'mean',
    'error_mitigated': 'mean',
    'energy_error_noisy': 'mean',
    'energy_error_mitigated': 'mean',
    'energy_improvement': 'mean',
    'depth_clifford': 'mean',
    'active_qubits_clifford': 'mean',
    'depth_transpiled': 'mean',
    'active_qubits_transpiled': 'mean',
    'circuit_creation_time': 'mean',
    'transpilation_time': 'mean',
    'clifford_ideal_estimation_time': 'mean',
    'clifford_noisy_estimation_time': 'mean',
    'clifford_mitigated_estimation_time': 'mean',
    'seed': 'count'
}).round(4)

agg_data.rename(columns={'seed': 'num_experiments'}, inplace=True)
agg_data = agg_data.reset_index()
agg_data['param_combo'] = agg_data.apply(lambda x: f"M{int(x['num_machines'])}_P{int(x['num_products'])}", axis=1)

print("Aggregated Data Summary (Mean values grouped by parameter combinations):")
print(agg_data[['param_combo', 'num_experiments', 'error_noisy', 'error_mitigated', 
                'energy_error_noisy', 'energy_error_mitigated', 'depth_clifford', 'active_qubits_clifford']])

# Aggregate by parameter combinations and factory
agg_by_factory = df_clean.groupby(['num_machines', 'num_products', 'factory']).agg({
    'ideal_energy': 'mean',
    'noisy_energy': 'mean',
    'mitigated_energy': 'mean',
    'error_noisy': 'mean',
    'error_mitigated': 'mean',
    'seed': 'count'
}).round(4)
agg_by_factory.rename(columns={'seed': 'count'}, inplace=True)
agg_by_factory = agg_by_factory.reset_index()

print("\nAggregated Data by Factory:")
print(agg_by_factory[['num_machines', 'num_products', 'factory', 'error_noisy', 'error_mitigated', 'count']])

Aggregated Data Summary (Mean values grouped by parameter combinations):
   param_combo  num_experiments  error_noisy  error_mitigated  \
0        M2_P6               13      65.7851          42.9294   
1        M2_P8               13      77.6264          53.0746   
2       M2_P10               12      82.4492          57.5681   
3       M2_P12               12      80.6025          60.1121   
4        M3_P6               12      52.4875          32.0302   
5        M3_P8               12      53.5804          24.8265   
6       M3_P10               14      69.9416          49.1321   
7       M3_P12               12      77.3292          50.2340   
8        M4_P6               15      49.5799          19.1583   
9        M4_P8               15      54.8140          19.8185   
10      M4_P10               15      66.7878          36.8280   
11      M4_P12               10      83.6179          70.2157   

    energy_error_noisy  energy_error_mitigated  depth_clifford  \
0             1

## 3. Error Analysis: Noisy vs Ideal

In [4]:
# 3.1 Noisy Error Percentage across parameter combinations
fig1 = px.bar(agg_data, x='param_combo', y='error_noisy', 
              color='param_combo', 
              title='Mean Noisy Error (%) across Problem Parameters',
              labels={'error_noisy': 'Error %', 'param_combo': 'Parameter Combination'},
              height=500)
fig1.update_layout(showlegend=False)
fig1.show()

# 3.2 Line plot showing error trends
agg_sorted = agg_data.sort_values(['num_machines', 'num_products'])
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Error vs Num Products (by Machines)", 
                                                      "Error vs Num Machines (by Products)"))

# Group by num_machines and plot error vs num_products
for machine in sorted(df_clean['num_machines'].unique()):
    subset = agg_sorted[agg_sorted['num_machines'] == machine]
    fig2.add_trace(go.Scatter(x=subset['num_products'], y=subset['error_noisy'],
                              name=f'M{int(machine)}', mode='lines+markers'),
                   row=1, col=1)

# Group by num_products and plot error vs num_machines
for product in sorted(df_clean['num_products'].unique()):
    subset = agg_sorted[agg_sorted['num_products'] == product]
    fig2.add_trace(go.Scatter(x=subset['num_machines'], y=subset['error_noisy'],
                              name=f'P{int(product)}', mode='lines+markers'),
                   row=1, col=2)

fig2.update_xaxes(title_text="Num Products", row=1, col=1)
fig2.update_xaxes(title_text="Num Machines", row=1, col=2)
fig2.update_yaxes(title_text="Error %", row=1, col=1)
fig2.update_yaxes(title_text="Error %", row=1, col=2)
fig2.update_layout(height=500, title_text="Noisy Error Trends")
fig2.show()

# 3.3 Distribution of noisy error by parameter combination (box plot)
fig3 = px.box(df_clean, x='param_combo', y='error_noisy',
              title='Distribution of Noisy Errors by Parameter Combination',
              labels={'error_noisy': 'Error %', 'param_combo': 'Parameter Combination'},
              height=500)
fig3.show()

## 4. Error Analysis: Mitigated vs Ideal

In [5]:
# 4.1 Comparison of Noisy vs Mitigated errors
fig4 = go.Figure()
fig4.add_trace(go.Bar(x=agg_data['param_combo'], y=agg_data['error_noisy'], 
                       name='Noisy Error %', marker_color='indianred'))
fig4.add_trace(go.Bar(x=agg_data['param_combo'], y=agg_data['error_mitigated'], 
                       name='Mitigated Error %', marker_color='lightsalmon'))
fig4.update_layout(title='Noisy vs Mitigated Error Comparison',
                   xaxis_title='Parameter Combination',
                   yaxis_title='Error %',
                   barmode='group',
                   height=500)
fig4.show()

# 4.2 Error improvement (how much mitigation helps)
agg_data['error_improvement_pct'] = agg_data['error_noisy'] - agg_data['error_mitigated']
fig5 = px.bar(agg_data, x='param_combo', y='error_improvement_pct',
              color='error_improvement_pct',
              title='Mitigation Effectiveness: Error Reduction (%)',
              labels={'error_improvement_pct': 'Error Reduction %', 'param_combo': 'Parameter Combination'},
              height=500,
              color_continuous_scale='RdYlGn')
fig5.show()

# 4.3 Mitigated error distribution
fig6 = px.box(df_clean, x='param_combo', y='error_mitigated',
              title='Distribution of Mitigated Errors by Parameter Combination',
              labels={'error_mitigated': 'Error %', 'param_combo': 'Parameter Combination'},
              height=500)
fig6.show()

# 4.4 Line plot: Error trends for both noisy and mitigated
fig7 = make_subplots(rows=1, cols=1, specs=[[{"secondary_y": False}]])
agg_sorted = agg_data.sort_values(['num_machines', 'num_products'])
for machine in sorted(agg_data['num_machines'].unique()):
    subset = agg_sorted[agg_sorted['num_machines'] == machine]
    fig7.add_trace(go.Scatter(x=subset['num_products'], y=subset['error_noisy'],
                              name=f'M{int(machine)} Noisy', mode='lines+markers',
                              line=dict(dash='solid')))
    fig7.add_trace(go.Scatter(x=subset['num_products'], y=subset['error_mitigated'],
                              name=f'M{int(machine)} Mitigated', mode='lines+markers',
                              line=dict(dash='dash')))

fig7.update_layout(title='Noisy vs Mitigated Error Trends',
                   xaxis_title='Num Products',
                   yaxis_title='Error %',
                   height=600)
fig7.show()

## 5. Circuit Complexity Analysis

In [6]:
# 5.1 Scatter plot: Depth vs Noisy Error
fig8 = px.scatter(agg_data, x='depth_clifford', y='error_noisy',
                  size='active_qubits_clifford', hover_data=['param_combo', 'num_experiments'],
                  title='Circuit Depth vs Noisy Error',
                  labels={'depth_clifford': 'Clifford Circuit Depth', 'error_noisy': 'Error %'},
                  height=500,
                  color_discrete_sequence=['steelblue'])
fig8.show()

# 5.2 Scatter plot: Depth vs Mitigated Error
fig9 = px.scatter(agg_data, x='depth_clifford', y='error_mitigated',
                  size='active_qubits_clifford', hover_data=['param_combo'],
                  title='Circuit Depth vs Mitigated Error',
                  labels={'depth_clifford': 'Clifford Circuit Depth', 'error_mitigated': 'Error %'},
                  height=500,
                  color_discrete_sequence=['forestgreen'])
fig9.show()

# 5.3 Scatter plot: Active Qubits vs Depth (colored by error)
fig10 = px.scatter(agg_data, x='active_qubits_clifford', y='depth_clifford',
                   size='error_noisy', color='error_noisy',
                   hover_data=['param_combo', 'error_mitigated'],
                   title='Active Qubits vs Circuit Depth (sized by Noisy Error)',
                   labels={'active_qubits_clifford': 'Active Qubits', 
                           'depth_clifford': 'Clifford Depth',
                           'error_noisy': 'Noisy Error %'},
                   height=500,
                   color_continuous_scale='Reds')
fig10.show()

# 5.4 Depth trends across parameters
fig11 = make_subplots(rows=1, cols=2, subplot_titles=("Depth vs Num Products", 
                                                       "Active Qubits vs Num Products"))
agg_sorted = agg_data.sort_values(['num_machines', 'num_products'])
for machine in sorted(agg_data['num_machines'].unique()):
    subset = agg_sorted[agg_sorted['num_machines'] == machine]
    fig11.add_trace(go.Scatter(x=subset['num_products'], y=subset['depth_clifford'],
                               name=f'M{int(machine)}', mode='lines+markers'),
                    row=1, col=1)
    fig11.add_trace(go.Scatter(x=subset['num_products'], y=subset['active_qubits_clifford'],
                               name=f'M{int(machine)}', mode='lines+markers', showlegend=False),
                    row=1, col=2)

fig11.update_xaxes(title_text="Num Products", row=1, col=1)
fig11.update_xaxes(title_text="Num Products", row=1, col=2)
fig11.update_yaxes(title_text="Depth", row=1, col=1)
fig11.update_yaxes(title_text="Active Qubits", row=1, col=2)
fig11.update_layout(height=500, title_text="Circuit Metrics Trends")
fig11.show()

# 5.5 Transpilation impact: Virtual vs Transpiled Depth
fig12 = go.Figure()
fig12.add_trace(go.Scatter(x=agg_data['param_combo'], y=agg_data['depth_clifford'],
                           name='Clifford Depth', mode='lines+markers'))
fig12.add_trace(go.Scatter(x=agg_data['param_combo'], y=agg_data['depth_transpiled'],
                           name='Transpiled Depth', mode='lines+markers'))
fig12.update_layout(title='Circuit Depth: Clifford vs Transpiled',
                    xaxis_title='Parameter Combination',
                    yaxis_title='Depth',
                    height=500)
fig12.show()

## 6. Performance Metrics by Problem Parameters

In [13]:
# 6.1 Energy levels across parameters - grouped bar chart
energy_comparison = agg_data[['param_combo', 'ideal_energy', 'noisy_energy', 'mitigated_energy']].copy()
energy_comparison['ideal_normalized'] = (energy_comparison['ideal_energy'] - energy_comparison['ideal_energy'].min()) / (energy_comparison['ideal_energy'].max() - energy_comparison['ideal_energy'].min())
energy_comparison['noisy_normalized'] = (energy_comparison['noisy_energy'] - energy_comparison['ideal_energy']) 
energy_comparison['mitigated_normalized'] = (energy_comparison['mitigated_energy'] - energy_comparison['ideal_energy'])

fig13 = make_subplots(rows=1, cols=2, 
                      subplot_titles=("Energy Deviation from Ideal (Absolute)", 
                                     "Energy Levels"),
                      specs=[[{"type": "bar"}, {"type": "bar"}]])

# Absolute deviations
fig13.add_trace(go.Bar(x=agg_data['param_combo'], 
                       y=agg_data['energy_error_noisy'],
                       name='Noisy Error', marker_color='indianred'),
                row=1, col=1)
fig13.add_trace(go.Bar(x=agg_data['param_combo'], 
                       y=agg_data['energy_error_mitigated'],
                       name='Mitigated Error', marker_color='lightsalmon'),
                row=1, col=1)

# Energy levels
fig13.add_trace(go.Bar(x=agg_data['param_combo'], 
                       y=agg_data['ideal_energy'],
                       name='Ideal', marker_color='green'),
                row=1, col=2)
fig13.add_trace(go.Bar(x=agg_data['param_combo'], 
                       y=agg_data['noisy_energy'],
                       name='Noisy', marker_color='red'),
                row=1, col=2)
fig13.add_trace(go.Bar(x=agg_data['param_combo'], 
                       y=agg_data['mitigated_energy'],
                       name='Mitigated', marker_color='orange'),
                row=1, col=2)

fig13.update_xaxes(title_text="Parameter Combination", row=1, col=1)
fig13.update_xaxes(title_text="Parameter Combination", row=1, col=2)
fig13.update_yaxes(title_text="Energy Error", row=1, col=1)
fig13.update_yaxes(title_text="Energy Value", row=1, col=2)
fig13.update_layout(height=500, title_text="Energy Performance Metrics")
fig13.show()

# 6.2 Energy trends: separate plots for each machine count
fig14 = make_subplots(rows=1, cols=len(sorted(agg_data['num_machines'].unique())),
                      subplot_titles=[f"Machines={int(m)}" for m in sorted(agg_data['num_machines'].unique())],
                      specs=[[{"secondary_y": False}] * len(sorted(agg_data['num_machines'].unique()))])

for idx, machine in enumerate(sorted(agg_data['num_machines'].unique()), 1):
    subset = agg_data[agg_data['num_machines'] == machine].sort_values('num_products')
    fig14.add_trace(go.Scatter(x=subset['num_products'], y=subset['ideal_energy'],
                               name='Ideal', mode='lines+markers'),
                    row=1, col=idx)
    fig14.add_trace(go.Scatter(x=subset['num_products'], y=subset['noisy_energy'],
                               name='Noisy', mode='lines+markers', showlegend=(idx==1)),
                    row=1, col=idx)
    fig14.add_trace(go.Scatter(x=subset['num_products'], y=subset['mitigated_energy'],
                               name='Mitigated', mode='lines+markers', showlegend=(idx==1)),
                    row=1, col=idx)

for idx in range(1, len(sorted(agg_data['num_machines'].unique())) + 1):
    fig14.update_xaxes(title_text="Num Products", row=1, col=idx)
    fig14.update_yaxes(title_text="Energy", row=1, col=1)

fig14.update_layout(height=500, title_text="Energy Trends by Problem Parameters")
fig14.show()

## 7. Mitigation Technique Comparison

In [14]:
# Check if we have multiple factories
factories = df_clean['factory'].unique()
print(f"Mitigation factories in dataset: {factories}")

# 7.1 Factory comparison: Error performance
fig15 = px.box(df_clean, x='factory', y='error_mitigated',
               color='factory',
               title='Mitigated Error Distribution by Factory Type',
               labels={'factory': 'Factory', 'error_mitigated': 'Error %'},
               height=500)
fig15.show()

# 7.2 Factory comparison: Noisy vs Mitigated error by parameter combo
fig16 = px.box(df_clean, x='param_combo', y='error_mitigated',
               color='factory',
               title='Mitigated Error by Parameter Combination and Factory',
               labels={'param_combo': 'Parameter Combination', 'error_mitigated': 'Error %'},
               height=500)
fig16.show()

# 7.3 Factory effectiveness: Error reduction percentage
if len(factories) > 1:
    factory_stats = df_clean.groupby('factory').agg({
        'error_noisy': 'mean',
        'error_mitigated': 'mean',
    }).reset_index()
    factory_stats['error_reduction'] = factory_stats['error_noisy'] - factory_stats['error_mitigated']
    factory_stats['reduction_pct'] = (factory_stats['error_reduction'] / factory_stats['error_noisy'] * 100).round(2)
    
    fig17 = make_subplots(rows=1, cols=2, subplot_titles=("Mean Errors by Factory", 
                                                           "Error Reduction %"))
    
    # Mean errors
    fig17.add_trace(go.Bar(x=factory_stats['factory'], y=factory_stats['error_noisy'],
                           name='Noisy', marker_color='indianred'),
                    row=1, col=1)
    fig17.add_trace(go.Bar(x=factory_stats['factory'], y=factory_stats['error_mitigated'],
                           name='Mitigated', marker_color='lightsalmon'),
                    row=1, col=1)
    
    # Reduction percentage
    colors = ['green' if x > 0 else 'red' for x in factory_stats['reduction_pct']]
    fig17.add_trace(go.Bar(x=factory_stats['factory'], y=factory_stats['reduction_pct'],
                           marker_color=colors, name='Reduction %', showlegend=False),
                    row=1, col=2)
    
    fig17.update_xaxes(title_text="Factory", row=1, col=1)
    fig17.update_xaxes(title_text="Factory", row=1, col=2)
    fig17.update_yaxes(title_text="Error %", row=1, col=1)
    fig17.update_yaxes(title_text="Reduction %", row=1, col=2)
    fig17.update_layout(height=500, title_text="Mitigation Technique Effectiveness")
    fig17.show()
    
    print("\nFactory Statistics:")
    print(factory_stats)

# 7.4 Factory performance by parameter combination
if len(factories) > 1:
    fig18 = px.scatter(df_clean, x='error_noisy', y='error_mitigated',
                       color='factory', hover_data=['param_combo', 'seed'],
                       title='Factory Performance: Noisy Error vs Mitigated Error',
                       labels={'error_noisy': 'Noisy Error %', 'error_mitigated': 'Mitigated Error %'},
                       height=500)
    fig18.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                               name='No Improvement', line=dict(dash='dash', color='gray')))
    fig18.update_layout(xaxis=dict(range=[0, 100]), yaxis=dict(range=[0, 100]))
    fig18.show()

Mitigation factories in dataset: ['linear' 'richardson' 'adaexp']



Factory Statistics:
      factory  error_noisy  error_mitigated  error_reduction  reduction_pct
0      adaexp    66.304716        32.896009        33.408707          50.39
1      linear    68.128968        56.394291        11.734677          17.22
2  richardson    66.606960        32.289358        34.317602          51.52


## 8. Computational Time Analysis

In [12]:
# 8.1 Estimation times comparison (the most critical time)
estimation_times = agg_data[['param_combo', 'clifford_ideal_estimation_time', 
                              'clifford_noisy_estimation_time', 'clifford_mitigated_estimation_time']].copy()

fig19 = go.Figure()
fig19.add_trace(go.Bar(x=estimation_times['param_combo'], 
                       y=estimation_times['clifford_ideal_estimation_time'],
                       name='Ideal Estimation Time'))
fig19.add_trace(go.Bar(x=estimation_times['param_combo'], 
                       y=estimation_times['clifford_noisy_estimation_time'],
                       name='Noisy Estimation Time'))
fig19.add_trace(go.Bar(x=estimation_times['param_combo'], 
                       y=estimation_times['clifford_mitigated_estimation_time'],
                       name='Mitigated Estimation Time'))
fig19.update_layout(title='Estimation Times by Result Type and Parameter',
                    xaxis_title='Parameter Combination',
                    yaxis_title='Time (seconds)',
                    barmode='group',
                    height=500)
fig19.show()

# 8.2 Total execution time: Circuit + Transpilation + Estimation
agg_data['total_time'] = (agg_data['circuit_creation_time'] + 
                          agg_data['transpilation_time'] + 
                          agg_data['clifford_ideal_estimation_time'])
agg_data['total_time_mitigated'] = (agg_data['circuit_creation_time'] + 
                                    agg_data['transpilation_time'] + 
                                    agg_data['clifford_mitigated_estimation_time'])

fig20 = go.Figure()
fig20.add_trace(go.Bar(x=agg_data['param_combo'], y=agg_data['total_time'],
                       name='Ideal Total Time', marker_color='lightblue'))
fig20.add_trace(go.Bar(x=agg_data['param_combo'], y=agg_data['total_time_mitigated'],
                       name='Mitigated Total Time', marker_color='lightcoral'))
fig20.update_layout(title='Total Execution Time (Circuit + Transpilation + Estimation)',
                    xaxis_title='Parameter Combination',
                    yaxis_title='Time (seconds)',
                    height=500)
fig20.show()

# 8.3 Time decomposition
fig21 = make_subplots(rows=1, cols=3, specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "bar"}]],
                      subplot_titles=("Circuit Creation Time", "Transpilation Time", 
                                     "Estimation Time Breakdown"))

for i, metric in enumerate([('circuit_creation_time', 'Circuit Creation'),
                            ('transpilation_time', 'Transpilation'),
                            ('clifford_ideal_estimation_time', 'Ideal Estimation')], 1):
    fig21.add_trace(go.Bar(x=agg_data['param_combo'], y=agg_data[metric[0]],
                           name=metric[1], showlegend=False),
                    row=1, col=i)
    fig21.update_xaxes(title_text="Parameter", row=1, col=i)
    if i == 1:
        fig21.update_yaxes(title_text="Time (s)", row=1, col=i)

fig21.update_layout(height=400, title_text="Time Decomposition")
fig21.show()

# 8.4 Estimation time scaling with problem complexity
fig22 = px.scatter(agg_data, x='num_products', y='clifford_ideal_estimation_time',
                   size='depth_clifford', color='error_noisy', hover_data=['param_combo'],
                   title='Estimation Time vs Problem Complexity',
                   labels={'num_products': 'Num Products', 
                           'clifford_ideal_estimation_time': 'Estimation Time (s)',
                           'error_noisy': 'Noisy Error %'},
                   height=500)
fig22.show()

# 8.5 Time statistics by factory (if applicable)
if len(factories) > 1:
    time_by_factory = df_clean.groupby('factory').agg({
        'clifford_ideal_estimation_time': ['mean', 'std'],
        'clifford_mitigated_estimation_time': ['mean', 'std'],
    }).round(4)
    print("\nEstimation Times by Factory:")
    print(time_by_factory)


Estimation Times by Factory:
           clifford_ideal_estimation_time          \
                                     mean     std   
factory                                             
adaexp                             2.6676  1.6761   
linear                             3.1401  2.7091   
richardson                         3.1463  2.7912   

           clifford_mitigated_estimation_time            
                                         mean       std  
factory                                                  
adaexp                              1111.5734  611.3115  
linear                               774.1411  495.7788  
richardson                           754.6487  496.8408  


## 9. Multi-parameter Analysis and Heatmaps

In [11]:
# Prepare data for heatmaps - pivot tables
# 9.1 Noisy Error Heatmap
noisy_error_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='error_noisy')
fig23 = px.imshow(noisy_error_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Error %"),
                  title='Noisy Error % Heatmap',
                  color_continuous_scale='Reds', text_auto='.2f', height=400)
fig23.show()

# 9.2 Mitigated Error Heatmap
mitigated_error_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='error_mitigated')
fig24 = px.imshow(mitigated_error_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Error %"),
                  title='Mitigated Error % Heatmap',
                  color_continuous_scale='Greens', text_auto='.2f', height=400)
fig24.show()

# 9.3 Error Improvement Heatmap
error_improvement_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='error_improvement_pct')
fig25 = px.imshow(error_improvement_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Improvement %"),
                  title='Error Improvement by Mitigation (%)',
                  color_continuous_scale='RdYlGn', text_auto='.2f', height=400)
fig25.show()

# 9.4 Circuit Depth Heatmap
depth_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='depth_clifford')
fig26 = px.imshow(depth_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Depth"),
                  title='Circuit Depth Heatmap',
                  color_continuous_scale='Blues', text_auto='.0f', height=400)
fig26.show()

# 9.5 Active Qubits Heatmap
qubits_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='active_qubits_clifford')
fig27 = px.imshow(qubits_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Active Qubits"),
                  title='Active Qubits Heatmap',
                  color_continuous_scale='Purples', text_auto='.0f', height=400)
fig27.show()

# 9.6 Energy Error (Absolute) Heatmap
energy_error_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='energy_error_noisy')
fig28 = px.imshow(energy_error_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Energy Error"),
                  title='Noisy Energy Error (Absolute Deviation) Heatmap',
                  color_continuous_scale='Oranges', text_auto='.0f', height=400)
fig28.show()

# 9.7 Estimation Time Heatmap
time_heatmap = agg_data.pivot(index='num_machines', columns='num_products', values='clifford_mitigated_estimation_time')
fig29 = px.imshow(time_heatmap, labels=dict(x="Num Products", y="Num Machines", color="Time (s)"),
                  title='Mitigated Estimation Time Heatmap',
                  color_continuous_scale='Viridis', text_auto='.0f', height=400)
fig29.show()

## 10. Summary Visualizations and Dashboard

In [7]:
# 10.1 Comprehensive multi-plot dashboard
fig30 = make_subplots(
    rows=3, cols=2,
    subplot_titles=("Error Comparison", "Energy Levels",
                    "Circuit Depth vs Error", "Estimation Time Scaling",
                    "Error Improvement Distribution", "Depth vs Active Qubits"),
    specs=[[{}, {}],
           [{}, {}],
           [{}, {}]]
)

# Plot 1: Error Comparison
fig30.add_trace(
    go.Bar(x=agg_data['param_combo'], y=agg_data['error_noisy'], 
           name='Noisy', marker_color='indianred'),
    row=1, col=1
)
fig30.add_trace(
    go.Bar(x=agg_data['param_combo'], y=agg_data['error_mitigated'], 
           name='Mitigated', marker_color='lightsalmon'),
    row=1, col=1
)

# Plot 2: Energy Levels
fig30.add_trace(
    go.Scatter(x=agg_data['param_combo'], y=agg_data['ideal_energy'],
               name='Ideal', mode='lines+markers', line=dict(color='green')),
    row=1, col=2
)
fig30.add_trace(
    go.Scatter(x=agg_data['param_combo'], y=agg_data['noisy_energy'],
               name='Noisy', mode='lines+markers', line=dict(color='red')),
    row=1, col=2
)
fig30.add_trace(
    go.Scatter(x=agg_data['param_combo'], y=agg_data['mitigated_energy'],
               name='Mitigated', mode='lines+markers', line=dict(color='orange')),
    row=1, col=2
)

# Plot 3: Depth vs Error
fig30.add_trace(
    go.Scatter(x=agg_data['depth_clifford'], y=agg_data['error_noisy'],
               mode='markers', marker=dict(size=10, color='red'),
               name='Noisy', showlegend=False),
    row=2, col=1
)
fig30.add_trace(
    go.Scatter(x=agg_data['depth_clifford'], y=agg_data['error_mitigated'],
               mode='markers', marker=dict(size=10, color='orange'),
               name='Mitigated', showlegend=False),
    row=2, col=1
)

# Plot 4: Estimation Time Scaling
agg_sorted = agg_data.sort_values('num_products')
fig30.add_trace(
    go.Scatter(x=agg_sorted['num_products'], y=agg_sorted['clifford_ideal_estimation_time'],
               mode='lines+markers', name='Ideal Time', line=dict(color='blue')),
    row=2, col=2
)
fig30.add_trace(
    go.Scatter(x=agg_sorted['num_products'], y=agg_sorted['clifford_mitigated_estimation_time'],
               mode='lines+markers', name='Mitigated Time', line=dict(color='purple')),
    row=2, col=2
)

# Plot 5: Error Improvement Distribution
colors = ['green' if x > 0 else 'red' for x in agg_data['error_improvement_pct']]
fig30.add_trace(
    go.Bar(x=agg_data['param_combo'], y=agg_data['error_improvement_pct'],
           marker_color=colors, name='Improvement', showlegend=False),
    row=3, col=1
)

# Plot 6: Depth vs Active Qubits
fig30.add_trace(
    go.Scatter(x=agg_data['active_qubits_clifford'], y=agg_data['depth_clifford'],
               mode='markers+text', text=agg_data['param_combo'],
               marker=dict(size=15, color=agg_data['error_noisy'], 
                          colorscale='Reds', showscale=True),
               name='Circuits', showlegend=False),
    row=3, col=2
)

# Update axes labels
fig30.update_xaxes(title_text="Parameter", row=1, col=1)
fig30.update_xaxes(title_text="Parameter", row=1, col=2)
fig30.update_xaxes(title_text="Depth", row=2, col=1)
fig30.update_xaxes(title_text="Num Products", row=2, col=2)
fig30.update_xaxes(title_text="Parameter", row=3, col=1)
fig30.update_xaxes(title_text="Active Qubits", row=3, col=2)

fig30.update_yaxes(title_text="Error %", row=1, col=1)
fig30.update_yaxes(title_text="Energy", row=1, col=2)
fig30.update_yaxes(title_text="Error %", row=2, col=1)
fig30.update_yaxes(title_text="Time (s)", row=2, col=2)
fig30.update_yaxes(title_text="Improvement %", row=3, col=1)
fig30.update_yaxes(title_text="Depth", row=3, col=2)

fig30.update_layout(height=1200, title_text="Comprehensive Analysis Dashboard", 
                    showlegend=True, hovermode='closest')
fig30.show()

# 10.2 Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print("\nOverall Error Statistics:")
print(f"  Mean Noisy Error: {agg_data['error_noisy'].mean():.2f}%")
print(f"  Mean Mitigated Error: {agg_data['error_mitigated'].mean():.2f}%")
print(f"  Mean Error Improvement: {agg_data['error_improvement_pct'].mean():.2f}%")

print("\nCircuit Complexity:")
print(f"  Min Depth: {agg_data['depth_clifford'].min():.0f}, Max: {agg_data['depth_clifford'].max():.0f}")
print(f"  Min Active Qubits: {agg_data['active_qubits_clifford'].min():.0f}, Max: {agg_data['active_qubits_clifford'].max():.0f}")

print("\nComputational Time:")
print(f"  Mean Circuit Creation Time: {agg_data['circuit_creation_time'].mean():.4f}s")
print(f"  Mean Transpilation Time: {agg_data['transpilation_time'].mean():.4f}s")
print(f"  Mean Ideal Estimation Time: {agg_data['clifford_ideal_estimation_time'].mean():.2f}s")
print(f"  Mean Mitigated Estimation Time: {agg_data['clifford_mitigated_estimation_time'].mean():.2f}s")

print("\nBest performing parameter combinations:")
best_by_error = agg_data.nsmallest(3, 'error_mitigated')[['param_combo', 'error_mitigated', 'depth_clifford']]
print(best_by_error)


SUMMARY STATISTICS

Overall Error Statistics:
  Mean Noisy Error: 67.88%
  Mean Mitigated Error: 42.99%
  Mean Error Improvement: 24.89%

Circuit Complexity:
  Min Depth: 1117, Max: 6373
  Min Active Qubits: 24, Max: 74

Computational Time:
  Mean Circuit Creation Time: 0.0383s
  Mean Transpilation Time: 1.2566s
  Mean Ideal Estimation Time: 3.14s
  Mean Mitigated Estimation Time: 849.23s

Best performing parameter combinations:
  param_combo  error_mitigated  depth_clifford
8       M4_P6          19.1583       2491.3333
9       M4_P8          19.8185       3358.5333
5       M3_P8          24.8265       2444.0000


## 11. Key Insights and Recommendations

In [10]:
# Detailed analysis and insights
print("\n" + "="*80)
print("DETAILED ANALYSIS AND INSIGHTS")
print("="*80)

# 1. Error analysis
print("\n1. ERROR ANALYSIS:")
worst_mitigated = agg_data.loc[agg_data['error_mitigated'].idxmax()]
best_mitigated = agg_data.loc[agg_data['error_mitigated'].idxmin()]
print(f"  Worst mitigated error: {worst_mitigated['param_combo']} ({worst_mitigated['error_mitigated']:.2f}%)")
print(f"  Best mitigated error: {best_mitigated['param_combo']} ({best_mitigated['error_mitigated']:.2f}%)")

# Mitigation effectiveness
positive_improvement = (agg_data['error_improvement_pct'] > 0).sum()
total = len(agg_data)
print(f"  Mitigation improved {positive_improvement}/{total} ({100*positive_improvement/total:.1f}%) of cases")

# 2. Circuit complexity vs error correlation
correlation_depth_noisy = agg_data['depth_clifford'].corr(agg_data['error_noisy'])
correlation_depth_mitigated = agg_data['depth_clifford'].corr(agg_data['error_mitigated'])
correlation_qubits_noisy = agg_data['active_qubits_clifford'].corr(agg_data['error_noisy'])
correlation_qubits_mitigated = agg_data['active_qubits_clifford'].corr(agg_data['error_mitigated'])

print("\n2. CORRELATIONS (Circuit Complexity vs Error):")
print(f"  Depth vs Noisy Error: {correlation_depth_noisy:.3f}")
print(f"  Depth vs Mitigated Error: {correlation_depth_mitigated:.3f}")
print(f"  Active Qubits vs Noisy Error: {correlation_qubits_noisy:.3f}")
print(f"  Active Qubits vs Mitigated Error: {correlation_qubits_mitigated:.3f}")

# 3. Problem parameter effects
print("\n3. PARAMETER EFFECTS:")
machines = sorted(agg_data['num_machines'].unique())
products = sorted(agg_data['num_products'].unique())

for m in machines:
    subset = agg_data[agg_data['num_machines'] == m]
    print(f"  Machines={int(m)}: Error range: {subset['error_mitigated'].min():.2f}% - {subset['error_mitigated'].max():.2f}%")

print()
for p in products:
    subset = agg_data[agg_data['num_products'] == p]
    print(f"  Products={int(p)}: Error range: {subset['error_mitigated'].min():.2f}% - {subset['error_mitigated'].max():.2f}%")

# 4. Factory effectiveness (if multiple factories)
if len(factories) > 1:
    print("\n4. MITIGATION FACTORY EFFECTIVENESS:")
    factory_effectiveness = df_clean.groupby('factory').agg({
        'error_noisy': ['mean', 'std'],
        'error_mitigated': ['mean', 'std'],
    }).round(2)
    print(factory_effectiveness)
    
    # Which factory has lowest mitigated error on average
    best_factory_idx = df_clean.groupby('factory')['error_mitigated'].mean().idxmin()
    print(f"  Best performing factory (lowest mitigated error): {best_factory_idx}")

# 5. Time scaling analysis
print("\n5. TIME SCALING ANALYSIS:")
print(f"  Correlation: Depth vs Estimation Time: {agg_data['depth_clifford'].corr(agg_data['clifford_mitigated_estimation_time']):.3f}")
print(f"  Correlation: Active Qubits vs Estimation Time: {agg_data['active_qubits_clifford'].corr(agg_data['clifford_mitigated_estimation_time']):.3f}")

# 6. Energy behavior
print("\n6. ENERGY BEHAVIOR:")
print(f"  Energy value range: [{agg_data['ideal_energy'].min():.2f}, {agg_data['ideal_energy'].max():.2f}]")
print(f"  Mean absolute energy error (noisy): {agg_data['energy_error_noisy'].mean():.2f}")
print(f"  Mean absolute energy error (mitigated): {agg_data['energy_error_mitigated'].mean():.2f}")

print("\n" + "="*80)


DETAILED ANALYSIS AND INSIGHTS

1. ERROR ANALYSIS:
  Worst mitigated error: M4_P12 (70.22%)
  Best mitigated error: M4_P6 (19.16%)
  Mitigation improved 12/12 (100.0%) of cases

2. CORRELATIONS (Circuit Complexity vs Error):
  Depth vs Noisy Error: 0.368
  Depth vs Mitigated Error: 0.354
  Active Qubits vs Noisy Error: 0.345
  Active Qubits vs Mitigated Error: 0.313

3. PARAMETER EFFECTS:
  Machines=2: Error range: 42.93% - 60.11%
  Machines=3: Error range: 24.83% - 50.23%
  Machines=4: Error range: 19.16% - 70.22%

  Products=6: Error range: 19.16% - 42.93%
  Products=8: Error range: 19.82% - 53.07%
  Products=10: Error range: 36.83% - 57.57%
  Products=12: Error range: 50.23% - 70.22%

4. MITIGATION FACTORY EFFECTIVENESS:
           error_noisy        error_mitigated       
                  mean    std            mean    std
factory                                             
adaexp           66.30  12.97           32.90  25.51
linear           68.13  14.67           56.39  20.52
